# Eclat — demo Google Colab

Chạy **Runtime → Run all** bằng Python kernel. Notebook cài Julia, chuẩn bị năm CSDL thật, chạy test, Groceries và Fashion-MNIST bằng Eclat của nhóm. Cần Internet và repository truy cập được từ Colab.

Mặc định Fashion dùng mẫu cân bằng 1.000 ảnh (100/lớp), K=32, seed=123 để demo nhanh. Đặt `FASHION_PROFILE="full"` để dựng giao dịch theo profile release đủ 70.000 ảnh; kết quả mẫu không thay thế kết quả báo cáo. Benchmark đầy đủ là tùy chọn riêng vì cần nhiều tiến trình. Mọi lệnh lỗi sẽ dừng notebook.


## 1. Cài Julia và lấy mã nguồn
Nếu chạy notebook trong checkout local, giữ nguyên thư mục hiện tại. Trên Colab, clone nhánh đã chứa notebook này. Nhánh cần được push trước khi dùng link Colab. Không đặt token vào notebook.

In [ ]:
from pathlib import Path
import os, subprocess, sys, shutil, tarfile, urllib.request, hashlib, json

REPO_URL = "https://github.com/NguyenVoTanDuy2006/DataMining-FIM-Project.git"
REPO_REF = "feature/complete-code"
FASHION_PROFILE = "100"  # "full": 70.000 ảnh, có thể mất nhiều thời gian/RAM hơn
RUN_BENCHMARKS = False  # True: chạy lại cả bốn nhóm benchmark
BENCHMARK_RUNS = 1      # đổi thành 5 để có median như thiết kế báo cáo

def run(*args):
    subprocess.run([str(a) for a in args], check=True)

if not Path("Project.toml").exists():
    checkout = Path("/content/DataMining-FIM-Project")
    if not checkout.exists():
        run("git", "clone", "--branch", REPO_REF, "--single-branch", REPO_URL, checkout)
    os.chdir(checkout)
assert Path("scripts/fashion_colab_demo.jl").is_file(), "Checkout chưa có code demo Colab"
if shutil.which("julia") is None:
    version = "1.13.3"
    base = "https://julialang-s3.julialang.org"
    archive = Path("/tmp/julia-demo.tar.gz")
    url = f"{base}/bin/linux/x64/1.13/julia-{version}-linux-x86_64.tar.gz"
    urllib.request.urlretrieve(url, archive)
    checksums = urllib.request.urlopen(f"{base}/bin/checksums/julia-{version}.sha256").read().decode()
    expected = next(line.split()[0] for line in checksums.splitlines()
                    if line.split()[-1] == f"julia-{version}-linux-x86_64.tar.gz")
    with archive.open("rb") as stream:
        assert hashlib.file_digest(stream, "sha256").hexdigest() == expected
    with tarfile.open(archive) as bundle:
        bundle.extractall("/tmp", filter="data")
    os.environ["PATH"] = f"/tmp/julia-{version}/bin:" + os.environ["PATH"]
os.environ["JULIA_NUM_THREADS"] = "1"
run("julia", "--version")
run("julia", "--project=.", "-e", "using Pkg; Pkg.instantiate()")


## 2. Dữ liệu và kiểm thử
Downloader xác minh SHA-256, giữ file hiện có nếu bị sai checksum và chuẩn hóa ID theo đúng benchmark. Tests so toàn bộ itemset/support với SPMF trên năm CSDL, sau đó chạy các test Fashion Python.

In [ ]:
run("julia", "--project=.", "scripts/prepare_data.jl")
run("julia", "--project=.", "test/runtests.jl")
run(sys.executable, "-m", "unittest", "discover", "-s", "test", "-p", "test_fashion_fim_*.py")
run(sys.executable, "src/applications/Fashion-Mnist/notebook_tools/validate_release_notebooks.py")


## 3. Groceries — top 10 luật theo lift
Bước chuẩn bị đã chạy pipeline Groceries: mã hóa giỏ, mining ở minsup 1%, lọc confidence 50% và xếp theo lift. Bảng thứ hai cho thấy 12 cấu hình ngưỡng. Đồng xuất hiện không chứng minh nhân quả.

In [ ]:
import csv, html
from IPython.display import display, Image, HTML

def show_csv(path):
    with open(path, newline="", encoding="utf-8") as stream:
        rows = list(csv.reader(stream))
    display(HTML("<table>" + "".join("<tr>" + "".join(
        "<td>" + html.escape(value) + "</td>" for value in row) + "</tr>"
        for row in rows) + "</table>"))

show_csv("results/groceries/top10_rules.csv")
show_csv("results/groceries/parameter_summary.csv")


## 4. Fashion-MNIST — ảnh → visual words → Eclat → diễn giải
Dùng lại các bước TC1–TC7: patch 4×4, pixel foreground >29, raw descriptor 16D, K-means K=32/seed=123 và binary BoVW. Hai backend chạy ở sáu ngưỡng, so toàn bộ output và audit support trực tiếp. Deadline 120 giây và giới hạn 100.000 itemsets áp dụng cho mỗi lần mine; nếu vượt giới hạn, demo dừng và không ghi manifest PASS.

Thời gian ở đây chỉ đo `mine` sau warm-up nhỏ, không phải số đo toàn tiến trình/RSS của báo cáo. Demo trực tiếp không cần systemd. Nếu chọn full, phần transaction dùng đúng profile release; các bước selection/interpretation ở đây chỉ phục vụ demo, không thay thế bộ bốn notebook release.

In [ ]:
before = set(Path("results/colab").glob("fashion_*/manifest.toml"))
run("julia", "--project=.", "scripts/fashion_colab_demo.jl", FASHION_PROFILE)
created = set(Path("results/colab").glob("fashion_*/manifest.toml")) - before
assert len(created) == 1, "Không tìm thấy đúng một run hoàn tất"
fashion_dir = created.pop().parent
show_csv(fashion_dir / "comparison.csv")
show_csv(fashion_dir / "patterns.csv")
for image in sorted(fashion_dir.glob("pattern_*.png")):
    print(image.name, "— hàng 1: centroid; hàng 2: patch thật từ ảnh hỗ trợ đầu tiên; dưới: ảnh chứa toàn bộ itemset")
    display(Image(filename=str(image)))
print("Diễn giải: xem pattern xuất hiện tập trung ở lớp nào; centroid là mẫu pixel 4×4,")
print("không mặc định tương ứng một bộ phận ngữ nghĩa. Cần người đọc review ý nghĩa mẫu.")


## 5. Benchmark — chạy lại tùy chọn
Bật `RUN_BENCHMARKS=True` ở ô đầu để chạy bốn dataset × bảy minsup với SPMF, Peak RSS, scalability và độ dài giao dịch. Ô này kiểm tra Java và tải JRE 25 có kiểm tra SHA-256 nếu Java mặc định quá cũ so với SPMF. JAR được tải từ SPMF chính thức; SHA-256 được lưu cùng run và metadata runner. Dữ liệu và phép đo mới nằm trong `results/colab/benchmarks/`, tách khỏi số liệu báo cáo đã commit. Một lần đo chỉ là demo; dùng năm lần để tổng hợp median.

In [ ]:
if RUN_BENCHMARKS:
    # SPMF hiện tại yêu cầu Java 25; không dùng Java 17/21 mặc định của Colab.
    probe = subprocess.run(["java", "-version"], capture_output=True, text=True) if shutil.which("java") else None
    import re
    version_match = re.search(r'version "(\d+)', probe.stderr) if probe else None
    if not version_match or int(version_match.group(1)) < 25:
        api = "https://api.adoptium.net/v3/assets/latest/25/hotspot?architecture=x64&image_type=jre&os=linux"
        asset = json.load(urllib.request.urlopen(urllib.request.Request(api, headers={"User-Agent": "curl/8.5.0"})))[0]["binary"]["package"]
        runtime_archive = Path("/tmp/eclat-jre.tar.gz")
        run("curl", "-fL", "--retry", "2", asset["link"], "-o", runtime_archive)
        with runtime_archive.open("rb") as stream:
            assert hashlib.file_digest(stream, "sha256").hexdigest() == asset["checksum"]
        runtime_dir = Path("/tmp/eclat-jre25")
        runtime_dir.mkdir(exist_ok=True)
        with tarfile.open(runtime_archive) as bundle:
            bundle.extractall(runtime_dir, filter="data")
        java_bin = next(runtime_dir.glob("*/bin/java")).parent
        os.environ["PATH"] = str(java_bin) + ":" + os.environ["PATH"]
    run("java", "-version")
    jar = Path("data/benchmark/tools/spmf.jar")
    jar.parent.mkdir(parents=True, exist_ok=True)
    if not jar.exists():
        temporary = jar.with_suffix(".download")
        urllib.request.urlretrieve("https://www.philippe-fournier-viger.com/spmf/spmf.jar", temporary)
        assert temporary.read_bytes().startswith(b"PK"), "SPMF download không phải JAR"
        temporary.rename(jar)
    for name in ("processes", "peak_ram", "scalability", "transaction_length"):
        dest = Path("results/colab/benchmarks") / name
        run("julia", "--project=.", f"scripts/benchmark_{name}.jl",
            "--runs", BENCHMARK_RUNS, "--output-dir", dest)
        plot = "plot_process_benchmarks.py" if name == "processes" else f"plot_{name}.py"
        run(sys.executable, f"scripts/{plot}", dest)
        show_csv(dest / "summary.csv")
        for image in sorted(dest.glob("*.png")):
            display(Image(filename=str(image)))
else:
    print("Chưa chạy lại benchmark. Bật RUN_BENCHMARKS=True để đo trên runtime hiện tại.")


## 6. Lưu kết quả
Tải archive để giữ comparison, galleries và metadata của demo. Không dùng thời gian đo trên Colab để thay số liệu báo cáo trên máy khác nếu chưa cập nhật phương pháp và phân tích.

In [ ]:
archive = shutil.make_archive("/tmp/eclat-colab-results", "zip", "results", "colab")
print("Kết quả:", archive)
try:
    from google.colab import files
except ImportError:
    pass
else:
    files.download(archive)
